# Nuclear Scaling Pipeline — v9 (Refactored)

**Four-stage segmentation + label generation + U-Net training in one notebook.**

This notebook merges two prior generations:
- **Segmentation methodology** ported from `Label_Generation_QC_v1` (validated this session): h_maxima watershed, multi-plane + DBSCAN + RANSAC droplet geometry, NPC-shell gating.
- **Pipeline / training infrastructure** from `Nuclear_Segmentation_v8.1`: `TrainingConfig`, `ProcessPoolExecutor` parallel patch generation, sentinel resume, U-Net, weighted BCE+Dice loss.

### Architecture (per FOV, per timepoint)
1. **Inventory** — full-FOV h_maxima watershed at a reference plane (+ optional 3-plane consensus). Expensive, runs 1–3× per FOV.
2. **Gating** — NPC-shell organization (RANSAC circle fit on puncta) + Membrane co-localization. Drops empty droplets and cap/clump artifacts. Cheap, per droplet.
3. **Per-droplet geometry** — cropped multi-plane detection + DBSCAN center-linking + RANSAC wall fit. Droplet extent at the nucleus best-Z. Moderate cost, gated droplets only.
4. **Label assembly + patch emission** — 4-class multi-label patches in the **channel-last `(H,W,4)`** training contract, plus a binary nucleus-mask hyperstack TIFF.

### Invariants (do not violate)
- **Channel separation:** droplet-extent path may clip/blur/close freely; NPC puncta detection reads the **RAW** NPC plane. Separate array copies.
- **Patch contract:** input `(H,W,3)` order **NLS, NPC, Membrane**; label `(H,W,4)` = [background, droplet, npc, nucleus]; saved as separate `img_*.npy` / `lab_*.npy`.
- **Early-timepoint exclusion:** assembly-phase timepoints (default t=0–1) are excluded at generation; the gate runs uniformly only where both anchors are reliable.


In [ ]:
# ============================================================
# 1. Core imports
# ============================================================
from dataclasses import dataclass, field
from pathlib import Path
import os, math, time, gc, sys

from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing

import numpy as np
import pandas as pd

import tifffile as tiff
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from scipy import ndimage
from skimage import filters, morphology, measure, segmentation
from skimage.morphology import h_maxima
from skimage.feature import peak_local_max
from sklearn.cluster import DBSCAN

import tensorflow as tf
from tensorflow.keras import layers, models

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow      :", tf.__version__)
print("Num GPUs        :", len(tf.config.list_physical_devices("GPU")))
print("CPU count       :", os.cpu_count())


## 2. Configuration

`PipelineConfig` extends the calibrated v8.1 `TrainingConfig` with the new
geometry/gating parameters. All segmentation constants calibrated on
`control_extract_1.1.tif`.

In [ ]:
# ============================================================
# 2. Configuration
# ============================================================
@dataclass
class PipelineConfig:
    # ---- Project paths ----
    data_root: Path = Path("/home/tdeibert/Data/Machine_Learning_Dev/")
    model_root: Path = None
    image_root: Path = None
    out_root:   Path = None
    image_filename: str = "control_extract_1.1.tif"

    # ---- Image metadata (T, Z, C, Y, X) ----
    pixel_size_um: float = 0.108
    n_channels:    int   = 3
    membrane_channel_idx: int = 0     # Ch0 Membrane (dim NE probe)
    nucleus_channel_idx:  int = 1     # Ch1 NLS
    npc_channel_idx:      int = 2     # Ch2 NPC

    # ---- Z handling ----
    z_floor: int = 6                  # exclude coverslip-artifact planes below this
    # Equatorial band where the droplet widest cross-section reliably sits:
    inventory_ref_z: int = 15
    consensus_z_offsets: tuple = (-1, 0, 1)   # 3-plane consensus around ref_z
    consensus_tol_frac: float = 0.10          # accept if counts agree within 10%

    # ---- Early-timepoint exclusion (assembly phase) ----
    # Both gate anchors fail at t=0-1: too few puncta for a shell fit, NE probe
    # too faint to co-localize. Excluded at generation; deferred to a future
    # assembly-phase model.
    generation_min_timepoint: int = 2

    # ---- Droplet detection (h_maxima watershed; geometry-only path) ----
    npc_clip_lo_pct: float = 1.0
    npc_clip_hi_pct: float = 80.0     # p1-p80 suppresses bright puncta
    droplet_blur_sigma: float = 8.0
    adaptive_block: int = 301         # odd
    adaptive_offset: float = -0.05
    closing_radius_px: int = 12       # bridges intermittent dark vacuoles
    h_depth: float = 15.0             # h_maxima prominence (replaces min_peak_distance)
    min_droplet_area_um2: float = 150.0
    droplet_min_circ: float = 0.70
    erosion_px: int = 10

    # ---- Per-droplet geometry (multi-plane + DBSCAN + RANSAC) ----
    geom_z_lo_offset: int = -4        # crop multi-plane search relative to nucleus best-z
    geom_z_hi_offset: int = 4
    drift_tolerance_um: float = 10.0  # DBSCAN eps in microns
    dbscan_min_planes: int = 2
    ransac_tol_px: float = 4.0
    ransac_n_iter: int = 200
    ransac_min_inlier_frac: float = 0.5
    wall_n_rays: int = 180
    wall_r_lo_frac: float = 0.55
    wall_r_hi_frac: float = 1.45
    wall_smooth_sigma: float = 2.0
    wall_strength_pct: float = 35.0

    # ---- NPC puncta (RAW channel; faithful intensities) ----
    npc_margin_px: int = 5            # annular zone half-width around nucleus edge
    npc_std_mult: float = 2.0         # threshold = mean + k*std of raw NPC interior

    # ---- NPC-shell gate (organization, not presence) ----
    gate_min_puncta: int = 8          # min puncta to attempt a shell fit
    gate_shell_tol_px: float = 6.0    # RANSAC residual tol for puncta-on-shell
    gate_shell_min_inlier_frac: float = 0.45
    gate_membrane_nn_radius_px: int = 6   # NPC punctum must have a membrane neighbor within R
    gate_membrane_min_coloc_frac: float = 0.30  # frac of puncta needing membrane support
    gate_membrane_k_std: float = 1.0  # membrane "present" = mean + k*std (dim probe)

    # ---- Nucleus (NLS) detection ----
    nucleus_block_size: object = None     # None = auto (~1/3 droplet diameter)
    nucleus_min_frac: float = 0.01
    nucleus_max_frac: float = 0.50        # backstop (NPC gate is primary empty filter)
    nucleus_min_circ: float = 0.40

    # ---- Patch extraction ----
    patch_size: int = 512
    patch_jitter_px: int = 128
    patches_per_droplet: int = 3
    min_label_fraction: float = 0.002     # on droplet channel
    timepoint_patch_weights: object = None

    # ---- Coverslip hard-negative sampling ----
    coverslip_negative_patches_per_t: int = 20
    coverslip_z_max: int = 6
    coverslip_neg_max_t: int = 5
    coverslip_min_brightness: float = 0.15

    # ---- Parallel ----
    max_parallel_workers: int = 10

    # ---- Model / training ----
    model_name: str = "unet_nuclear_scaling_v9"
    num_classes: int = 4
    batch_size: int = 2
    epochs: int = 50
    learning_rate: float = 1e-4
    validation_fraction: float = 0.20
    seed: int = 42
    use_augmentation: bool = True
    loss_class_weights: tuple = (0.3, 0.5, 5.0, 4.0)  # bg, droplet, npc, nucleus
    val_timepoints: object = None

    def __post_init__(self):
        if self.model_root is None: self.model_root = self.data_root / "Models"
        if self.image_root is None: self.image_root = self.data_root / "Images"
        if self.out_root   is None: self.out_root   = self.data_root / "Outputs"

    # ---- Derived paths ----
    @property
    def image_file(self):      return self.image_root / self.image_filename
    @property
    def training_root(self):   return self.out_root / f"training_patches_{self.model_name}"
    @property
    def image_patch_dir(self): return self.training_root / "images"
    @property
    def label_patch_dir(self): return self.training_root / "labels"
    @property
    def qc_dir(self):          return self.out_root / f"qc_{self.model_name}"
    @property
    def best_model_path(self): return self.model_root / f"{self.model_name}_best.keras"
    @property
    def final_model_path(self):return self.model_root / f"{self.model_name}_final.keras"

    def min_droplet_area_px(self):
        return self.min_droplet_area_um2 / (self.pixel_size_um ** 2)

    def patches_for_timepoint(self, t):
        if self.timepoint_patch_weights is None:
            return self.patches_per_droplet
        w = list(self.timepoint_patch_weights)
        wt = w[t] if t < len(w) else 1.0
        return max(1, round(self.patches_per_droplet * wt))


cfg = PipelineConfig()

# Timepoint weights: late (assembled-nucleus) timepoints dominate.
# t0-t1 weights are present but those timepoints are EXCLUDED at generation
# via cfg.generation_min_timepoint (assembly phase deferred to a future model).
cfg.timepoint_patch_weights = (0.25, 0.25, 0.5, 0.5, 1.0, 2.0, 3.0, 4.0, 4.0, 5.0)
cfg.val_timepoints = [2, 5, 8]
cfg.max_parallel_workers = int(os.environ.get("SLURM_CPUS_PER_TASK", 10))

for p in [cfg.data_root, cfg.model_root, cfg.image_root, cfg.out_root,
          cfg.training_root, cfg.image_patch_dir, cfg.label_patch_dir, cfg.qc_dir]:
    p.mkdir(parents=True, exist_ok=True)

# Channel / class constants
MEM_CH, NUC_CH, NPC_CH = cfg.membrane_channel_idx, cfg.nucleus_channel_idx, cfg.npc_channel_idx
PIXEL_SIZE_UM = cfg.pixel_size_um
PATCH_SIZE    = cfg.patch_size
NUM_CHANNELS  = cfg.n_channels
NUM_CLASSES   = cfg.num_classes
CLASS_BACKGROUND, CLASS_DROPLET, CLASS_NPC, CLASS_NUCLEUS = 0, 1, 2, 3
CLASS_NAMES = ["Background", "Droplet", "NPC", "Nucleus"]

print("image_file              :", cfg.image_file)
print("training_root           :", cfg.training_root)
print("generation_min_timepoint:", cfg.generation_min_timepoint)
print("inventory_ref_z         :", cfg.inventory_ref_z)
print("max_parallel_workers    :", cfg.max_parallel_workers)
print("min_droplet_area_px     :", round(cfg.min_droplet_area_px()))
print("patches/timepoint t0-9  :", [cfg.patches_for_timepoint(t) for t in range(10)])


## 3. Histogram-safe IO

`extract_plane` always returns a fresh float32 copy so nothing downstream can
mutate the source hyperstack or another class's view. `clip_histogram` is used
**only** on the droplet-detection path.

In [ ]:
# ============================================================
# 3. Histogram-safe channel access
# ============================================================
def load_memmap_tiff(path):
    """Open the hyperstack as a read-only memmap (T, Z, C, Y, X)."""
    return tiff.memmap(str(path))

def extract_plane(hyperstack, t, z, c):
    """Single 2D plane as a fresh float32 COPY (callers may clip/blur freely)."""
    return np.asarray(hyperstack[t, z, c], dtype=np.float32).copy()

def clip_histogram(img, lo_pct, hi_pct):
    """Percentile-clip + 0-1 normalise a COPY. Droplet-detection path ONLY."""
    work = img.astype(np.float32, copy=True)
    lo, hi = np.percentile(work, [lo_pct, hi_pct])
    work = np.clip(work, lo, hi)
    rng = hi - lo
    return np.zeros_like(work) if rng <= 0 else (work - lo) / rng

def _circularity(region):
    p = region.perimeter
    return 0.0 if p <= 0 else float(4.0 * np.pi * region.area / (p * p))

def normalize_channel(ch2d):
    """0-1 percentile normalise for model input patches."""
    ch = ch2d.astype(np.float32)
    lo, hi = np.percentile(ch, [1, 99.8])
    if hi <= lo:
        return np.zeros_like(ch)
    return np.clip((ch - lo) / (hi - lo), 0, 1)

def z_in_focus_range(z, n_z, cfg=cfg):
    """Exclude coverslip-artifact planes below the floor."""
    return cfg.z_floor <= z < n_z


## 4. Droplet geometry — shared primitives

This section is the validated geometry core. In the eventual package refactor it
becomes `droplet_geometry.py`, imported by both label generation and the
downstream ER radial analysis.

- `detect_droplets_npc_watershed` — h_maxima seeding + vacuole closing (Stage 1).
- `fit_circle_ransac` — **shared** RANSAC primitive, used for both droplet-wall
  fitting (Stage 3) and NPC-shell gating (Stage 2).
- `detect_droplets_multiplane` / `link_droplets_dbscan` — drift-tolerant linking.
- `extract_wall_points_radial` / `fit_droplet_circle_at_plane` — per-plane extent.

In [ ]:
# ============================================================
# 4a. Stage-1 detector: h_maxima watershed (geometry-only)
# ============================================================
def detect_droplets_npc_watershed(npc_plane, cfg=cfg):
    """
    Detect droplets from one NPC plane. Geometry-only: pixel values are NOT
    preserved. h_maxima seeding (depth-based) keeps one seed per droplet while
    preserving genuinely-touching droplets; closing bridges dark vacuoles.

    Returns list of dicts: {label, mask(eroded bool), bbox, centroid, area, circ}
    """
    img_norm = clip_histogram(npc_plane, cfg.npc_clip_lo_pct, cfg.npc_clip_hi_pct)
    img_blur = filters.gaussian(img_norm, sigma=cfg.droplet_blur_sigma)

    local_thresh = filters.threshold_local(img_blur, block_size=cfg.adaptive_block,
                                            offset=cfg.adaptive_offset)
    fg = img_blur > local_thresh
    fg = morphology.binary_closing(fg, morphology.disk(cfg.closing_radius_px))  # bridge vacuole
    fg = ndimage.binary_fill_holes(fg)
    min_area = int(cfg.min_droplet_area_px())
    fg = morphology.remove_small_objects(fg, min_size=min_area)

    dist = ndimage.distance_transform_edt(fg)
    maxima = h_maxima(dist, cfg.h_depth)              # depth-based seeds
    markers = measure.label(maxima)
    if markers.max() == 0:
        return []

    ws = segmentation.watershed(-dist, markers, mask=fg)

    droplets = []
    selem = morphology.disk(cfg.erosion_px)
    for region in measure.regionprops(ws):
        if region.area < min_area:
            continue
        circ = _circularity(region)
        if circ < cfg.droplet_min_circ:
            continue
        region_mask = ndimage.binary_fill_holes(ws == region.label)
        eroded = morphology.binary_erosion(region_mask, selem)
        if not eroded.any():
            continue
        ys, xs = np.where(eroded)
        droplets.append({
            "label":    int(region.label),
            "mask":     eroded,
            "bbox":     (int(ys.min()), int(xs.min()), int(ys.max())+1, int(xs.max())+1),
            "centroid": (float(ys.mean()), float(xs.mean())),
            "area":     int(eroded.sum()),
            "circ":     round(circ, 3),
        })
    return droplets


In [ ]:
# ============================================================
# 4b. Shared RANSAC circle primitive
#     Used for BOTH droplet-wall fitting and NPC-shell gating.
# ============================================================
def _circle_from_3(p1, p2, p3):
    ax, ay = p1; bx, by = p2; cx, cy = p3
    d = 2.0 * (ax*(by-cy) + bx*(cy-ay) + cx*(ay-by))
    if abs(d) < 1e-9:
        return None
    a2, b2, c2 = ax*ax+ay*ay, bx*bx+by*by, cx*cx+cy*cy
    ux = (a2*(by-cy) + b2*(cy-ay) + c2*(ay-by)) / d
    uy = (a2*(cx-bx) + b2*(ax-cx) + c2*(bx-ax)) / d
    return ux, uy, float(np.hypot(ux-ax, uy-ay))

def _fit_circle_kasa(pts):
    x, y = pts[:, 0], pts[:, 1]
    A = np.c_[x, y, np.ones(len(x))]
    b = x*x + y*y
    sol, *_ = np.linalg.lstsq(A, b, rcond=None)
    cx, cy = sol[0]/2.0, sol[1]/2.0
    r = np.sqrt(max(sol[2] + cx*cx + cy*cy, 0.0))
    return cx, cy, r

def fit_circle_ransac(pts, tol_px=4.0, n_iter=200, min_inlier_frac=0.5, rng=None):
    """
    RANSAC circle fit to Nx2 (x, y). Returns (cx, cy, r, inlier_mask) or None.
    Tolerant of missing arcs (touching neighbours) and stray points.
    Shared primitive: droplet wall (many points) and NPC shell (fewer points).
    """
    rng = rng or np.random.default_rng(0)
    n = len(pts)
    if n < 3:
        return None
    best_inliers, best_circle = None, None
    for _ in range(n_iter):
        idx = rng.choice(n, 3, replace=False)
        circ = _circle_from_3(pts[idx[0]], pts[idx[1]], pts[idx[2]])
        if circ is None:
            continue
        cx, cy, r = circ
        resid = np.abs(np.hypot(pts[:, 0]-cx, pts[:, 1]-cy) - r)
        inliers = resid < tol_px
        if best_inliers is None or inliers.sum() > best_inliers.sum():
            best_inliers, best_circle = inliers, circ
    if best_inliers is None or best_inliers.sum() < max(3, int(min_inlier_frac*n)):
        return None
    cx, cy, r = _fit_circle_kasa(pts[best_inliers])      # refit on inliers
    resid = np.abs(np.hypot(pts[:, 0]-cx, pts[:, 1]-cy) - r)
    return cx, cy, r, resid < tol_px


In [ ]:
# ============================================================
# 4c. Multi-plane detection + DBSCAN center-linking (drift-tolerant)
# ============================================================
def detect_droplets_multiplane(hyperstack, t, z_range, cfg=cfg):
    """Run the watershed detector on each z; return flat list of detections."""
    dets = []
    for z in z_range:
        npc = extract_plane(hyperstack, t, z, cfg.npc_channel_idx)
        for d in detect_droplets_npc_watershed(npc, cfg=cfg):
            cy, cx = d["centroid"]
            dets.append({"z": int(z), "x": float(cx), "y": float(cy),
                         "r": float(np.sqrt(d["area"]/np.pi)),
                         "area": int(d["area"]), "circ": float(d["circ"])})
    return dets

def link_droplets_dbscan(dets, cfg=cfg):
    """
    Cluster detections by (x, y) across z with DBSCAN. eps = drift tolerance
    (um) / pixel size. Each cluster is one physical droplet tracked through z.
    Returns list of tracks (each a list of detection dicts sorted by z).
    """
    if not dets:
        return []
    xy = np.array([[d["x"], d["y"]] for d in dets])
    eps_px = cfg.drift_tolerance_um / cfg.pixel_size_um
    labels = DBSCAN(eps=eps_px, min_samples=cfg.dbscan_min_planes).fit_predict(xy)
    tracks = []
    for lab in sorted(set(labels)):
        if lab == -1:
            continue
        members = [dets[i] for i in range(len(dets)) if labels[i] == lab]
        members.sort(key=lambda d: d["z"])
        tracks.append(members)
    return tracks


In [ ]:
# ============================================================
# 4d. Per-plane wall extraction + circle fit at a target plane
# ============================================================
def extract_wall_points_radial(npc_plane, center_xy, r_prior, cfg=cfg):
    """
    Cast rays from center; wall = steepest interior->exterior intensity fall in
    [r_lo, r_hi]*r_prior. Rays through touching neighbours give weak edges and
    are dropped by the strength percentile. Returns Nx2 (x, y). Geometry-only.
    """
    cx, cy = center_xy
    H, W = npc_plane.shape
    img = filters.gaussian(clip_histogram(npc_plane, cfg.npc_clip_lo_pct, cfg.npc_clip_hi_pct),
                           cfg.wall_smooth_sigma)
    radii = np.arange(r_prior*cfg.wall_r_lo_frac, r_prior*cfg.wall_r_hi_frac, 1.0)
    pts, strengths = [], []
    for theta in np.linspace(0, 2*np.pi, cfg.wall_n_rays, endpoint=False):
        dx, dy = np.cos(theta), np.sin(theta)
        xs, ys = cx + radii*dx, cy + radii*dy
        ok = (xs >= 0) & (xs < W) & (ys >= 0) & (ys < H)
        if ok.sum() < 5:
            continue
        prof = ndimage.map_coordinates(img, [ys[ok], xs[ok]], order=1)
        grad = np.gradient(prof)
        j = int(np.argmin(grad))                  # steepest fall = wall
        edge_r = radii[ok][j]
        pts.append((cx + edge_r*dx, cy + edge_r*dy))
        strengths.append(-grad[j])
    if not pts:
        return np.empty((0, 2))
    pts, strengths = np.array(pts), np.array(strengths)
    keep = strengths >= np.percentile(strengths, cfg.wall_strength_pct)
    return pts[keep]

def fit_droplet_circle_at_plane(hyperstack, t, z_target, seed_center_xy, r_prior, cfg=cfg):
    """
    Recover the droplet circle at z_target: wall points seeded from a (linked)
    center, then RANSAC. Returns (cx, cy, r, wall_pts, inlier_mask) or None.
    """
    npc = extract_plane(hyperstack, t, z_target, cfg.npc_channel_idx)
    wall = extract_wall_points_radial(npc, seed_center_xy, r_prior, cfg=cfg)
    if len(wall) < 3:
        return None
    fit = fit_circle_ransac(wall, tol_px=cfg.ransac_tol_px, n_iter=cfg.ransac_n_iter,
                            min_inlier_frac=cfg.ransac_min_inlier_frac)
    if fit is None:
        return None
    cx, cy, r, inliers = fit
    return cx, cy, r, wall, inliers

def circle_to_mask(cx, cy, r, shape):
    """Rasterise a filled circle to a boolean mask of the given (H, W) shape."""
    H, W = shape
    yy, xx = np.ogrid[:H, :W]
    return (xx - cx)**2 + (yy - cy)**2 <= r*r


## 5. Nucleus (NLS) and NPC puncta detection

Ported unchanged from the validated methodology. The threshold-sweep confirmed
the NLS detector is healthy when given a correct droplet interior. NPC puncta
detection reads the **RAW** NPC plane (channel-separation invariant).

In [ ]:
# ============================================================
# 5a. Nucleus (NLS) detection — adaptive with per-droplet Otsu fallback
# ============================================================
def detect_nucleus_adaptive(nls_crop, droplet_mask_crop, cfg=cfg):
    """
    Detect nucleus interior from the NLS crop of one droplet.
    Primary: adaptive local threshold within the droplet.
    Fallback: per-droplet Otsu on interior values.
    No nucleus hallucinated -> returns (empty, 'none') for early/empty cases.
    Returns (mask bool, method in {'adaptive','otsu','none'}).
    """
    nls = nls_crop.astype(np.float32, copy=True)
    droplet_area = int(droplet_mask_crop.sum())
    empty = np.zeros_like(droplet_mask_crop, dtype=bool)
    if droplet_area == 0:
        return empty, "none"

    if cfg.nucleus_block_size is None:
        diameter = 2.0 * np.sqrt(droplet_area / np.pi)
        bs = max(3, int(diameter / 3))
        bs = bs + 1 if bs % 2 == 0 else bs
    else:
        bs = cfg.nucleus_block_size

    def _clean_and_gate(mask):
        mask = mask & droplet_mask_crop
        mask = morphology.remove_small_objects(mask, min_size=64)
        mask = ndimage.binary_fill_holes(mask)
        frac = mask.sum() / droplet_area
        if not (cfg.nucleus_min_frac <= frac <= cfg.nucleus_max_frac):
            return None
        lbl = measure.label(mask)
        props = measure.regionprops(lbl)
        if not props:
            return None
        biggest = max(props, key=lambda r: r.area)
        if _circularity(biggest) < cfg.nucleus_min_circ:
            return None
        return lbl == biggest.label

    try:
        local_t = filters.threshold_local(nls, block_size=bs)
        cand = _clean_and_gate(nls > local_t)
        if cand is not None:
            return cand, "adaptive"
    except Exception:
        pass
    try:
        vals = nls[droplet_mask_crop]
        if vals.size and vals.max() > vals.min():
            cand = _clean_and_gate(nls > filters.threshold_otsu(vals))
            if cand is not None:
                return cand, "otsu"
    except Exception:
        pass
    return empty, "none"


In [ ]:
# ============================================================
# 5b. NPC puncta detection — RAW channel, nucleus-boundary anchored
# ============================================================
def detect_npc_puncta(npc_crop_raw, nucleus_mask_crop, droplet_mask_crop, cfg=cfg):
    """
    NPC puncta in an annular zone straddling the nucleus edge.
    npc_crop_raw MUST be the RAW NPC plane (channel-separation invariant) -
    the clipped droplet-detection array has the puncta signal removed.
    threshold = mean + k*std of RAW NPC over the droplet interior.
    """
    if nucleus_mask_crop.sum() == 0:
        return np.zeros_like(nucleus_mask_crop, dtype=bool)
    npc = npc_crop_raw.astype(np.float32, copy=True)
    selem = morphology.disk(cfg.npc_margin_px)
    outer = morphology.binary_dilation(nucleus_mask_crop, selem)
    inner = morphology.binary_erosion(nucleus_mask_crop, selem)
    zone = outer & ~inner
    interior_vals = npc[droplet_mask_crop]
    if interior_vals.size == 0:
        return np.zeros_like(nucleus_mask_crop, dtype=bool)
    thresh = interior_vals.mean() + cfg.npc_std_mult * interior_vals.std()
    return (npc > thresh) & zone


## 6. Stage-2 gate — NPC-shell organization + Membrane co-localization

The gate tests **organization, not presence**. A presence-only check passes
cap artifacts and antibody clumps. Two complementary geometric tests:

- **Shell organization (primary):** the shared `fit_circle_ransac` primitive is
  run on the NPC-punctum point cloud. A coherent low-residual circle of sensible
  radius = nuclear envelope; no consensus = scattered cap/clump signal.
- **Membrane co-localization (secondary):** true NPC sits on the NE, which the
  dim Membrane channel also marks. Each punctum needs a membrane neighbor within
  R; a minimum fraction must be supported.

Both must pass. Early timepoints (t < `generation_min_timepoint`) never reach
the gate — they are excluded at generation.

In [ ]:
# ============================================================
# 6. NPC-shell gate (organization, not presence)
# ============================================================
def _punctum_centroids(npc_puncta_mask):
    """Centroids (x, y) of connected NPC puncta in crop coordinates."""
    lbl = measure.label(npc_puncta_mask)
    return np.array([[r.centroid[1], r.centroid[0]]
                     for r in measure.regionprops(lbl)], dtype=float)

def gate_npc_shell(npc_puncta_mask, cfg=cfg):
    """
    Shell-organization test. Returns (passed: bool, info: dict).
    Puncta must admit a coherent RANSAC circle fit (envelope), not scatter.
    """
    cents = _punctum_centroids(npc_puncta_mask)
    info = {"n_puncta": len(cents), "shell_r": None, "shell_inliers": 0}
    if len(cents) < cfg.gate_min_puncta:
        return False, info
    fit = fit_circle_ransac(cents, tol_px=cfg.gate_shell_tol_px,
                            n_iter=cfg.ransac_n_iter,
                            min_inlier_frac=cfg.gate_shell_min_inlier_frac)
    if fit is None:
        return False, info
    cx, cy, r, inliers = fit
    info["shell_r"], info["shell_inliers"] = float(r), int(inliers.sum())
    passed = inliers.sum() >= max(3, int(cfg.gate_shell_min_inlier_frac * len(cents)))
    return passed, info

def gate_membrane_coloc(npc_puncta_mask, mem_crop_raw, droplet_mask_crop, cfg=cfg):
    """
    Membrane co-localization test. Returns (passed: bool, info: dict).
    Each NPC punctum must have membrane signal (mean + k*std, dim probe) within
    gate_membrane_nn_radius_px; a minimum fraction of puncta must be supported.
    """
    cents = _punctum_centroids(npc_puncta_mask)
    info = {"coloc_frac": 0.0}
    if len(cents) == 0:
        return False, info
    mem = mem_crop_raw.astype(np.float32, copy=True)
    interior = mem[droplet_mask_crop]
    if interior.size == 0:
        return False, info
    mem_thresh = interior.mean() + cfg.gate_membrane_k_std * interior.std()
    mem_present = mem > mem_thresh
    # dilate membrane-present mask by the NN radius; a punctum is supported if
    # its centroid falls within the dilated membrane region.
    mem_dil = morphology.binary_dilation(mem_present,
                                         morphology.disk(cfg.gate_membrane_nn_radius_px))
    H, W = mem.shape
    supported = 0
    for x, y in cents:
        xi, yi = int(round(x)), int(round(y))
        if 0 <= yi < H and 0 <= xi < W and mem_dil[yi, xi]:
            supported += 1
    frac = supported / len(cents)
    info["coloc_frac"] = float(frac)
    return frac >= cfg.gate_membrane_min_coloc_frac, info

def gate_droplet(npc_puncta_mask, mem_crop_raw, droplet_mask_crop, cfg=cfg):
    """
    Full Stage-2 gate: shell-organization AND membrane co-localization.
    Returns (passed: bool, info: dict). A droplet passes only if both fire.
    """
    shell_ok, shell_info = gate_npc_shell(npc_puncta_mask, cfg=cfg)
    mem_ok, mem_info = gate_membrane_coloc(npc_puncta_mask, mem_crop_raw,
                                           droplet_mask_crop, cfg=cfg)
    info = {**shell_info, **mem_info, "shell_ok": shell_ok, "membrane_ok": mem_ok}
    return (shell_ok and mem_ok), info


## 7. Label assembly — channel-LAST `(H,W,4)`

`build_label_stack_hwc` composes the 4-class multi-label target in the
**training contract orientation** (channel-last). The droplet channel uses the
RANSAC circle fit; nucleus is a subset of droplet; NPC puncta come from the raw
channel.

In [ ]:
# ============================================================
# 7. Label assembly (channel-LAST, training contract)
# ============================================================
def build_label_stack_hwc(shape, droplet_masks, nucleus_masks, npc_masks):
    """
    Compose 4-channel multi-label target as (H, W, 4) float32:
        ch0 Background = NOT droplet
        ch1 Droplet    = solid interior (INCLUDES nucleus footprint)
        ch2 NPC        = puncta on nuclear envelope
        ch3 Nucleus    = nucleus interior
    droplet_masks/nucleus_masks/npc_masks are FULL-FRAME boolean arrays.
    """
    H, W = shape
    droplet = np.zeros((H, W), bool)
    npc     = np.zeros((H, W), bool)
    nucleus = np.zeros((H, W), bool)
    for dm in droplet_masks:
        droplet |= dm
    for nm in nucleus_masks:
        nucleus |= nm
    for pm in npc_masks:
        npc |= pm
    droplet |= nucleus                       # nucleus lies inside droplet
    background = ~droplet
    stack = np.stack([background, droplet, npc, nucleus], axis=-1)  # (H, W, 4)
    return stack.astype(np.float32)

def collapse_to_integer_hwc(stack_hwc):
    """Collapse (H,W,4) multi-label to integer map. Priority Nuc>NPC>Droplet>BG."""
    H, W, _ = stack_hwc.shape
    out = np.zeros((H, W), dtype=np.uint8)
    out[stack_hwc[..., CLASS_DROPLET] > 0] = CLASS_DROPLET
    out[stack_hwc[..., CLASS_NPC]     > 0] = CLASS_NPC
    out[stack_hwc[..., CLASS_NUCLEUS] > 0] = CLASS_NUCLEUS
    return out


## 8. Patch extraction + IO (training contract)

Input patches are `(H,W,3)` channel-last in order **NLS, NPC, Membrane** via
`build_input_patch`; labels are `(H,W,4)`. Saved as separate `img_*.npy` /
`lab_*.npy` — exactly what `make_dataset` consumes. Sentinel files support
resume across compute windows.

In [ ]:
# ============================================================
# 8. Patch extraction + IO
# ============================================================
def build_input_patch(mem_patch, nuc_patch, npc_patch):
    """(H, W, 3) float32 in [0,1]. Channel order: NLS, NPC, Membrane."""
    return np.stack([normalize_channel(nuc_patch),
                     normalize_channel(npc_patch),
                     normalize_channel(mem_patch)], axis=-1).astype('float32')

def _safe_crop_2d(arr, cy, cx, size):
    """Centred crop with zero-padding when the window exceeds the image."""
    half = size // 2
    H, W = arr.shape[-2:]
    r0, c0 = int(round(cy)) - half, int(round(cx)) - half
    r1, c1 = r0 + size, c0 + size
    pr0, pc0 = max(0, -r0), max(0, -c0)
    sr0, sc0 = max(0, r0), max(0, c0)
    sr1, sc1 = min(H, r1), min(W, c1)
    out = np.zeros((size, size), dtype=arr.dtype)
    out[pr0:pr0+(sr1-sr0), pc0:pc0+(sc1-sc0)] = arr[sr0:sr1, sc0:sc1]
    return out

def _safe_crop_hwc(arr_hwc, cy, cx, size):
    """Centred crop of an (H, W, C) array with zero-padding."""
    half = size // 2
    H, W, C = arr_hwc.shape
    r0, c0 = int(round(cy)) - half, int(round(cx)) - half
    r1, c1 = r0 + size, c0 + size
    pr0, pc0 = max(0, -r0), max(0, -c0)
    sr0, sc0 = max(0, r0), max(0, c0)
    sr1, sc1 = min(H, r1), min(W, c1)
    out = np.zeros((size, size, C), dtype=arr_hwc.dtype)
    out[pr0:pr0+(sr1-sr0), pc0:pc0+(sc1-sc0), :] = arr_hwc[sr0:sr1, sc0:sc1, :]
    return out

def extract_input_patch(hyperstack, t, z, cy, cx, cfg=cfg):
    """Build the (H, W, 3) NLS/NPC/Membrane input patch from a plane."""
    mem = _safe_crop_2d(extract_plane(hyperstack, t, z, cfg.membrane_channel_idx), cy, cx, cfg.patch_size)
    nuc = _safe_crop_2d(extract_plane(hyperstack, t, z, cfg.nucleus_channel_idx),  cy, cx, cfg.patch_size)
    npc = _safe_crop_2d(extract_plane(hyperstack, t, z, cfg.npc_channel_idx),      cy, cx, cfg.patch_size)
    return build_input_patch(mem, nuc, npc)

def jitter_center(cy, cx, jitter_px, H, W, patch_size, rng):
    half = patch_size // 2
    if jitter_px and jitter_px > 0:
        cy += int(rng.integers(-jitter_px, jitter_px + 1))
        cx += int(rng.integers(-jitter_px, jitter_px + 1))
    cy = int(np.clip(cy, half, H - half))
    cx = int(np.clip(cx, half, W - half))
    return cy, cx

# ---- sentinel resume ----
def get_completed_timepoints(cfg=cfg):
    completed = set()
    for flag in cfg.training_root.glob('t???_complete.flag'):
        try:
            completed.add(int(flag.stem.split('_')[0][1:]))
        except (ValueError, IndexError):
            pass
    return completed

def write_timepoint_sentinel(cfg, t, n_patches):
    (cfg.training_root / f't{t:03d}_complete.flag').write_text(f't={t} patches={n_patches}')

def clear_timepoint_sentinel(cfg, t):
    f = cfg.training_root / f't{t:03d}_complete.flag'
    if f.exists():
        f.unlink()

def list_patch_files(cfg=cfg):
    img = sorted(cfg.image_patch_dir.glob('img_*.npy'))
    lab = sorted(cfg.label_patch_dir.glob('lab_*.npy'))
    return img, lab


## 9. Stage orchestration — `process_timepoint`

Ties the four stages together for one timepoint. This is the worker body run by
the parallel executor. Each worker opens its own read-only memmap.

Flow: inventory at ref-z (+ consensus) → per droplet: nucleus best-z scan,
NPC puncta on raw channel, **gate**, per-droplet RANSAC geometry → assemble
label plane → emit jittered patches. Empty/gated-out droplets never produce
patches. Accumulates a binary nucleus-mask hyperstack for the QC TIFF.

In [ ]:
# ============================================================
# 9a. Nucleus best-Z scan for one droplet (cheap, on crop)
# ============================================================
def nucleus_best_z_for_droplet(hyperstack, t, droplet, z_candidates, cfg=cfg):
    """
    Scan z_candidates for the plane with the largest valid nucleus in this
    droplet's bbox. Returns (best_z, best_nuc_mask_crop, method) or (None,...,'none').
    """
    r0, c0, r1, c1 = droplet["bbox"]
    dm = droplet["mask"][r0:r1, c0:c1]
    best = (None, np.zeros_like(dm), "none", 0)
    for z in z_candidates:
        nls = extract_plane(hyperstack, t, z, cfg.nucleus_channel_idx)[r0:r1, c0:c1]
        nuc, method = detect_nucleus_adaptive(nls, dm, cfg=cfg)
        area = int(nuc.sum())
        if area > best[3]:
            best = (int(z), nuc, method, area)
    return best[0], best[1], best[2]

# ============================================================
# 9b. Full per-timepoint pipeline (worker body)
# ============================================================
def process_timepoint(args):
    """
    Top-level worker (picklable). args = (t, image_path_str, cfg, seed).
    Returns (summary_rows, nucleus_mask_stack_for_t [Z, Y, X] uint8).
    """
    t, image_path_str, cfg, worker_seed = args
    import numpy as _np

    hs = tiff.memmap(image_path_str)
    n_t, n_z, n_c, H, W = hs.shape
    rng = _np.random.default_rng(worker_seed)
    summary_rows = []
    nuc_stack_t = _np.zeros((n_z, H, W), dtype=_np.uint8)

    valid_zs = [z for z in range(n_z) if z_in_focus_range(z, n_z, cfg=cfg)]
    if not valid_zs:
        write_timepoint_sentinel(cfg, t, 0)
        return summary_rows, nuc_stack_t

    # ---- Stage 1: inventory at ref-z (+ 3-plane consensus) ----
    ref_z = cfg.inventory_ref_z if cfg.inventory_ref_z in valid_zs else valid_zs[len(valid_zs)//2]
    inv = detect_droplets_npc_watershed(extract_plane(hs, t, ref_z, cfg.npc_channel_idx), cfg=cfg)
    n_ref = len(inv)
    consensus_counts = []
    for dz in cfg.consensus_z_offsets:
        zc = ref_z + dz
        if zc == ref_z or zc not in valid_zs:
            continue
        consensus_counts.append(len(detect_droplets_npc_watershed(
            extract_plane(hs, t, zc, cfg.npc_channel_idx), cfg=cfg)))
    consensus_ok = True
    if consensus_counts and n_ref > 0:
        spread = (max([n_ref]+consensus_counts) - min([n_ref]+consensus_counts)) / n_ref
        consensus_ok = spread <= cfg.consensus_tol_frac

    if n_ref == 0:
        write_timepoint_sentinel(cfg, t, 0)
        return summary_rows, nuc_stack_t

    # geometry multi-plane search range around the equatorial band
    z_geom = [z for z in range(ref_z + cfg.geom_z_lo_offset, ref_z + cfg.geom_z_hi_offset + 1)
              if z in valid_zs]

    n_patches_t = cfg.patches_for_timepoint(t)
    droplet_masks_full, nucleus_masks_full, npc_masks_full = [], [], []
    accepted = 0

    for did, droplet in enumerate(inv):
        r0, c0, r1, c1 = droplet["bbox"]
        dm_crop = droplet["mask"][r0:r1, c0:c1]

        # ---- nucleus best-z (presence + focus combined) ----
        best_z, nuc_crop, method = nucleus_best_z_for_droplet(hs, t, droplet, valid_zs, cfg=cfg)
        if best_z is None or nuc_crop.sum() == 0:
            continue   # no functional nucleus -> not a labeling target

        # ---- NPC puncta on RAW channel at best_z ----
        npc_raw_crop = extract_plane(hs, t, best_z, cfg.npc_channel_idx)[r0:r1, c0:c1]
        mem_raw_crop = extract_plane(hs, t, best_z, cfg.membrane_channel_idx)[r0:r1, c0:c1]
        npc_crop = detect_npc_puncta(npc_raw_crop, nuc_crop, dm_crop, cfg=cfg)

        # ---- Stage 2: gate (shell organization + membrane co-loc) ----
        passed, ginfo = gate_droplet(npc_crop, mem_raw_crop, dm_crop, cfg=cfg)
        if not passed:
            continue

        # ---- Stage 3: per-droplet RANSAC geometry at best_z ----
        cy0, cx0 = droplet["centroid"]
        r_prior = float(np.sqrt(droplet["area"] / np.pi))
        fit = fit_droplet_circle_at_plane(hs, t, best_z, (cx0, cy0), r_prior, cfg=cfg)
        if fit is None:
            # fallback: use the inventory eroded mask if RANSAC fails
            drop_full = _np.zeros((H, W), bool)
            drop_full[r0:r1, c0:c1] = dm_crop
        else:
            fcx, fcy, fr, _, _ = fit
            drop_full = circle_to_mask(fcx, fcy, fr, (H, W))

        # ---- place crops into full-frame masks ----
        nuc_full = _np.zeros((H, W), bool); nuc_full[r0:r1, c0:c1] = nuc_crop
        npc_full = _np.zeros((H, W), bool); npc_full[r0:r1, c0:c1] = npc_crop
        nuc_full &= drop_full   # nucleus stays inside droplet

        droplet_masks_full.append(drop_full)
        nucleus_masks_full.append(nuc_full)
        npc_masks_full.append(npc_full)
        nuc_stack_t[best_z] |= nuc_full.astype(_np.uint8)
        accepted += 1

        summary_rows.append({
            "t": t, "droplet_id": did, "best_z": best_z, "nucleus_method": method,
            "shell_r": ginfo.get("shell_r"), "n_puncta": ginfo.get("n_puncta"),
            "coloc_frac": round(ginfo.get("coloc_frac", 0.0), 3),
            "consensus_ok": consensus_ok,
        })

    if accepted == 0:
        write_timepoint_sentinel(cfg, t, 0)
        return summary_rows, nuc_stack_t

    # ---- Stage 4: assemble label plane + emit patches ----
    # Patches are emitted per droplet at that droplet's best_z, using the
    # full-frame label stack masked to that droplet's neighbourhood.
    target_hwc = build_label_stack_hwc((H, W), droplet_masks_full,
                                       nucleus_masks_full, npc_masks_full)
    patch_id = 0
    for row, drop_full, nuc_full in zip(summary_rows, droplet_masks_full, nucleus_masks_full):
        did, best_z = row["droplet_id"], row["best_z"]
        ys, xs = np.where(drop_full)
        if len(ys) == 0:
            continue
        cy_base, cx_base = int(ys.mean()), int(xs.mean())
        for _ in range(n_patches_t):
            cy, cx = jitter_center(cy_base, cx_base, cfg.patch_jitter_px,
                                   H, W, cfg.patch_size, rng)
            y_patch = _safe_crop_hwc(target_hwc, cy, cx, cfg.patch_size)
            if _np.mean(y_patch[..., CLASS_DROPLET] > 0) < cfg.min_label_fraction:
                continue
            x_patch = extract_input_patch(hs, t, best_z, cy, cx, cfg=cfg)
            stem = f"t{t:03d}_z{best_z:03d}_d{did:04d}_y{cy:04d}_x{cx:04d}_p{patch_id:06d}"
            _np.save(cfg.image_patch_dir / f"img_{stem}.npy", x_patch.astype('float32'))
            _np.save(cfg.label_patch_dir / f"lab_{stem}.npy", y_patch.astype('float32'))
            patch_id += 1

    write_timepoint_sentinel(cfg, t, patch_id)
    return summary_rows, nuc_stack_t


## 10. Parallel patch generation

One worker per timepoint via `ProcessPoolExecutor` with **fork** context
(required for Jupyter `__main__`). Early (assembly-phase) timepoints below
`generation_min_timepoint` are skipped. Resume-aware via sentinel files. The
binary nucleus-mask hyperstack TIFF is assembled from all workers' returns.

In [ ]:
# ============================================================
# 10. Parallel driver
# ============================================================
def build_training_patches_parallel(cfg=cfg, overwrite=False):
    """
    Generate patches for all eligible timepoints in parallel.
    Eligible = generation_min_timepoint <= t < n_t (assembly phase excluded).
    Writes per-timepoint patches + a combined nucleus-mask hyperstack TIFF.
    """
    hs = tiff.memmap(str(cfg.image_file))
    n_t, n_z, n_c, H, W = hs.shape
    del hs  # workers reopen their own memmap

    eligible = [t for t in range(cfg.generation_min_timepoint, n_t)]
    if overwrite:
        for t in eligible:
            clear_timepoint_sentinel(cfg, t)
    else:
        done = get_completed_timepoints(cfg)
        eligible = [t for t in eligible if t not in done]

    print(f"Timepoints to process: {eligible}  (excluded t<{cfg.generation_min_timepoint})")
    if not eligible:
        print("Nothing to do (all complete or none eligible).")
        return

    args_list = [(t, str(cfg.image_file), cfg, cfg.seed + t) for t in eligible]
    nuc_hyperstack = np.zeros((n_t, n_z, H, W), dtype=np.uint8)
    all_rows = []

    ctx = multiprocessing.get_context("fork")
    n_workers = min(cfg.max_parallel_workers, len(args_list))
    with ProcessPoolExecutor(max_workers=n_workers, mp_context=ctx) as ex:
        futures = {ex.submit(process_timepoint, a): a[0] for a in args_list}
        for fut in as_completed(futures):
            t = futures[fut]
            rows, nuc_stack_t = fut.result()
            nuc_hyperstack[t] = nuc_stack_t
            all_rows.extend(rows)
            print(f"  t={t}: {len(rows)} accepted droplets", flush=True)

    # ---- nucleus-mask hyperstack TIFF (ROIs -> applied back for quantification) ----
    tiff_path = cfg.qc_dir / f"nucleus_mask_hyperstack_{cfg.model_name}.tif"
    tiff.imwrite(str(tiff_path), nuc_hyperstack, imagej=True, metadata={"axes": "TZYX"})
    print(f"\nNucleus-mask hyperstack -> {tiff_path}  shape={nuc_hyperstack.shape}")

    if all_rows:
        df = pd.DataFrame(all_rows)
        csv_path = cfg.qc_dir / f"generation_summary_{cfg.model_name}.csv"
        df.to_csv(csv_path, index=False)
        print(f"Summary -> {csv_path}")
        print(df.groupby("t").size().rename("accepted_droplets"))
    return


# ---- run it (uncomment to execute on a compute node) ----
# build_training_patches_parallel(cfg, overwrite=False)


## 11. Patch QC preview

Spot-check a random sample of emitted patches: NLS input channel (top) and the
collapsed 4-class label (bottom). Confirms channel orientation and label sanity
before committing to a full training run.

In [ ]:
# ============================================================
# 11. Patch QC
# ============================================================
_LABEL_CMAP = ListedColormap(["#2d004b", "#1f78b4", "#ffd700", "#1b7837"])  # bg/drop/npc/nuc

def preview_patches(n=8, cfg=cfg, seed=None):
    img_paths, lab_paths = list_patch_files(cfg)
    if not img_paths:
        print("No patches found. Run build_training_patches_parallel first.")
        return
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(img_paths), size=min(n, len(img_paths)), replace=False)
    fig, ax = plt.subplots(2, len(idx), figsize=(2.6*len(idx), 5.4))
    if len(idx) == 1:
        ax = ax[:, None]
    for j, i in enumerate(idx):
        x = np.load(img_paths[i]); y = np.load(lab_paths[i])
        ax[0, j].imshow(x[..., 0], cmap="gray")            # NLS channel
        ax[0, j].set_title(img_paths[i].name.replace("img_", "")[:22], fontsize=6)
        ax[1, j].imshow(collapse_to_integer_hwc(y), cmap=_LABEL_CMAP, vmin=0, vmax=3)
        ax[0, j].axis("off"); ax[1, j].axis("off")
    plt.suptitle("Patch QC — NLS input (top) | 4-class label (bottom)")
    plt.tight_layout(); plt.show()

# preview_patches(8)


## 12. Training — datasets, U-Net, loss

Carried through unchanged from v8.1 (the patch contract is identical, so the
training half does not change). Multi-label sigmoid head; weighted BCE + Dice;
per-class Dice metrics with `val_npc_dice` as the checkpoint monitor.

In [ ]:
# ============================================================
# 12a. TF dataset — multi-label sigmoid targets (H, W, 4)
# ============================================================
def augment_pair(image, label):
    n_img_ch = image.shape[-1]
    combined = tf.concat([image, label], axis=-1)
    combined = tf.image.random_flip_left_right(combined)
    combined = tf.image.random_flip_up_down(combined)
    k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)
    combined = tf.image.rot90(combined, k=k)
    return combined[..., :n_img_ch], combined[..., n_img_ch:]

def load_npy_pair_py(img_path, lab_path):
    if isinstance(img_path, bytes): img_path = img_path.decode('utf-8')
    if isinstance(lab_path, bytes): lab_path = lab_path.decode('utf-8')
    return np.load(img_path).astype('float32'), np.load(lab_path).astype('float32')

def tf_load_npy_pair(img_path, lab_path):
    x, y = tf.numpy_function(load_npy_pair_py, [img_path, lab_path], [tf.float32, tf.float32])
    x.set_shape((cfg.patch_size, cfg.patch_size, cfg.n_channels))
    y.set_shape((cfg.patch_size, cfg.patch_size, cfg.num_classes))
    return x, y

def make_dataset(img_paths, lab_paths, batch_size=None, shuffle=True, augment=False):
    batch_size = batch_size or cfg.batch_size
    ds = tf.data.Dataset.from_tensor_slices((list(map(str, img_paths)), list(map(str, lab_paths))))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(img_paths), seed=cfg.seed, reshuffle_each_iteration=True)
    ds = ds.map(tf_load_npy_pair, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        ds = ds.map(augment_pair, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

def build_train_val_datasets(cfg=cfg):
    img_paths, lab_paths = list_patch_files(cfg)
    if len(img_paths) != len(lab_paths):
        raise ValueError(f"Image/label count mismatch: {len(img_paths)} vs {len(lab_paths)}")
    if not img_paths:
        raise ValueError("No patches found. Run generation first.")
    val_t = list(cfg.val_timepoints) if cfg.val_timepoints is not None else []
    val_t_strs = [f"_t{t:03d}_" for t in val_t]
    train_img, train_lab, val_img, val_lab = [], [], [], []
    for ip, lp in zip(img_paths, lab_paths):
        if any(s in ip.name for s in val_t_strs):
            val_img.append(ip); val_lab.append(lp)
        else:
            train_img.append(ip); train_lab.append(lp)
    if not val_img:
        raise ValueError(f"No patches matched val_timepoints={val_t}.")
    print(f"Train patches: {len(train_img)}   Val patches: {len(val_img)}")
    train_ds = make_dataset(train_img, train_lab, cfg.batch_size, shuffle=True,  augment=cfg.use_augmentation)
    val_ds   = make_dataset(val_img,   val_lab,   cfg.batch_size, shuffle=False, augment=False)
    return train_ds, val_ds, train_img, val_img

# train_ds, val_ds, train_img_paths, val_img_paths = build_train_val_datasets(cfg)


In [ ]:
# ============================================================
# 12b. U-Net (multi-label sigmoid)
# ============================================================
def conv_block(x, filters, dropout_rate=0.0):
    x = layers.Conv2D(filters, 3, padding='same')(x); x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    x = layers.Conv2D(filters, 3, padding='same')(x); x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    if dropout_rate > 0.0:
        x = layers.SpatialDropout2D(dropout_rate)(x)
    return x

def encoder_block(x, filters, dropout_rate=0.0):
    c = conv_block(x, filters, dropout_rate=dropout_rate)
    return c, layers.MaxPooling2D((2, 2))(c)

def decoder_block(x, skip, filters, dropout_rate=0.0):
    x = layers.Conv2DTranspose(filters, 2, strides=2, padding='same')(x)
    x = layers.Concatenate()([x, skip])
    return conv_block(x, filters, dropout_rate=dropout_rate)

def build_unet(input_shape=None, num_classes=None):
    input_shape = input_shape or (cfg.patch_size, cfg.patch_size, cfg.n_channels)
    num_classes = num_classes or cfg.num_classes
    inputs = layers.Input(shape=input_shape)
    c1, p1 = encoder_block(inputs, 32)
    c2, p2 = encoder_block(p1, 64)
    c3, p3 = encoder_block(p2, 128)
    c4, p4 = encoder_block(p3, 256)
    bn = conv_block(p4, 512, dropout_rate=0.5)
    d1 = decoder_block(bn, c4, 256, dropout_rate=0.3)
    d2 = decoder_block(d1, c3, 128, dropout_rate=0.3)
    d3 = decoder_block(d2, c2, 64)
    d4 = decoder_block(d3, c1, 32)
    outputs = layers.Conv2D(num_classes, 1, activation='sigmoid')(d4)  # independent binary channels
    return models.Model(inputs, outputs, name=cfg.model_name)

# model = build_unet(); model.summary()


In [ ]:
# ============================================================
# 12c. Loss, metrics, training
# ============================================================
_CLASS_WEIGHTS = tf.constant(list(cfg.loss_class_weights), dtype=tf.float32)

def weighted_binary_dice_loss(y_true, y_pred, smooth=1e-6):
    axes = [0, 1, 2]
    inter = tf.reduce_sum(y_true * y_pred, axis=axes)
    denom = tf.reduce_sum(y_true + y_pred, axis=axes)
    dice = (2.0*inter + smooth) / (denom + smooth)
    return 1.0 - tf.reduce_sum(_CLASS_WEIGHTS * dice) / tf.reduce_sum(_CLASS_WEIGHTS)

def weighted_binary_ce_loss(y_true, y_pred):
    eps = 1e-7
    yp = tf.clip_by_value(y_pred, eps, 1.0 - eps)
    bce = -(y_true*tf.math.log(yp) + (1.0-y_true)*tf.math.log(1.0-yp))
    return tf.reduce_mean(bce * _CLASS_WEIGHTS)

def combined_weighted_loss(y_true, y_pred):
    return weighted_binary_ce_loss(y_true, y_pred) + weighted_binary_dice_loss(y_true, y_pred)

def _class_dice(y_true, y_pred, ch, smooth=1e-6):
    yt = y_true[..., ch]; yp = tf.cast(y_pred[..., ch] > 0.5, tf.float32)
    inter = tf.reduce_sum(yt*yp); denom = tf.reduce_sum(yt) + tf.reduce_sum(yp)
    return (2.0*inter + smooth) / (denom + smooth)

def dice_coefficient(y_true, y_pred, smooth=1e-6):
    yp = tf.cast(y_pred > 0.5, tf.float32); axes = [0,1,2]
    inter = tf.reduce_sum(y_true*yp, axis=axes); denom = tf.reduce_sum(y_true+yp, axis=axes)
    return tf.reduce_mean((2.0*inter + smooth)/(denom + smooth))

def npc_dice(y_true, y_pred):     return _class_dice(y_true, y_pred, CLASS_NPC)
def nucleus_dice(y_true, y_pred): return _class_dice(y_true, y_pred, CLASS_NUCLEUS)
def droplet_dice(y_true, y_pred): return _class_dice(y_true, y_pred, CLASS_DROPLET)

def compile_and_train(model, train_ds, val_ds, cfg=cfg):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=cfg.learning_rate),
        loss=combined_weighted_loss,
        metrics=[dice_coefficient, npc_dice, nucleus_dice, droplet_dice])
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(str(cfg.best_model_path), monitor="val_npc_dice",
                                           save_best_only=True, mode='max', verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_npc_dice", factor=0.5, patience=4,
                                             min_delta=1e-4, mode='max', verbose=1),
        tf.keras.callbacks.EarlyStopping(monitor="val_npc_dice", patience=10,
                                         restore_best_weights=True, mode='max', verbose=1),
    ]
    history = model.fit(train_ds, validation_data=val_ds, epochs=cfg.epochs, callbacks=callbacks)
    model.save(cfg.final_model_path)
    return history

# history = compile_and_train(model, train_ds, val_ds, cfg)


## 13. SLURM / tunnel reference

Standard interactive Ampere allocation:
```
srun --partition=amperenodes --gres=gpu:1 --ntasks=1 --cpus-per-task=8 --mem=32G --time=4:00:00 --pty bash
~/.vscode/cli/code tunnel --name star-forge
```
Patch generation parallelism uses `SLURM_CPUS_PER_TASK` workers. For
fire-and-forget batch generation, wrap `build_training_patches_parallel` in an
`sbatch` script once the methodology is locked.

### Build-order reminder (from the refactor plan)
1. Verify channel order in `build_input_patch` (NLS/NPC/Membrane) — **done, baked in here**.
2. Validate `droplet_geometry` on known droplets/timepoints.
3. Validate the Stage-2 gate against a known empty droplet and a cytoplasmic clump.
4. Small generation run → confirm patches load through `make_dataset`.
5. Carry over remaining QC: early-timepoint NLS, full-FOV consensus, drift.
